In [ ]:
!nvidia-smi | head -20

In [ ]:
%%writefile bench.cu
#include <stdio.h>
#include <stdint.h>
#include <stdlib.h>
#include <cuda_runtime.h>

#define PHI_UP 1696631u
#define W3_STRIDE 34u
#define W3_BASE   72u

typedef struct __attribute__((packed)) {
    uint64_t addr;
    uint32_t morton, hilbert;
    uint8_t  lane, slice, audit, flags;
    uint32_t phi;
} Coord; /* 24B */

__device__ const uint8_t lut[16]={0,3,4,5,1,2,7,6,14,13,8,9,15,12,11,10};

__global__ void kernel(Coord *c, uint32_t n) {
    uint32_t i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    uint64_t a = c[i].addr;
    uint32_t sc = 1u<<20;
    uint32_t t = (uint32_t)(((a&(sc-1u))*1696631ULL)>>20);
    uint16_t x = t&0x3FFu, y = (t>>10)&0x3FFu;
    uint32_t rx=x,ry=y;
    rx=(rx|(rx<<8))&0x00FF00FFu; rx=(rx|(rx<<4))&0x0F0F0F0Fu;
    rx=(rx|(rx<<2))&0x33333333u; rx=(rx|(rx<<1))&0x55555555u;
    ry=(ry|(ry<<8))&0x00FF00FFu; ry=(ry|(ry<<4))&0x0F0F0F0Fu;
    ry=(ry|(ry<<2))&0x33333333u; ry=(ry|(ry<<1))&0x55555555u;
    c[i].morton  = rx|(ry<<1);
    c[i].hilbert = ((c[i].morton>>4)<<4)|lut[c[i].morton&0xF];
    c[i].lane    = (uint8_t)(c[i].hilbert % 54u);
    c[i].audit   = ((a%17u)==1u) ? 0u : 1u;
}

void bench(uint32_t N, uint32_t tpb, const char *label) {
    Coord *h = (Coord*)malloc(N*sizeof(Coord));
    Coord *d; cudaMalloc(&d, N*sizeof(Coord));
    /* 34n+1 addresses */
    for (uint32_t i=0;i<N;i++) h[i].addr = (uint64_t)34*i + 1;
    cudaMemcpy(d,h,N*sizeof(Coord),cudaMemcpyHostToDevice);
    uint32_t blk=(N+tpb-1)/tpb;
    /* warmup */
    kernel<<<blk,tpb>>>(d,N); cudaDeviceSynchronize();
    /* timed */
    cudaEvent_t t0,t1;
    cudaEventCreate(&t0); cudaEventCreate(&t1);
    cudaEventRecord(t0);
    for(int r=0;r<10;r++) kernel<<<blk,tpb>>>(d,N);
    cudaEventRecord(t1); cudaDeviceSynchronize();
    float ms=0; cudaEventElapsedTime(&ms,t0,t1);
    double mops=(double)N*10/(ms/1000.0)/1e6;
    /* verify */
    cudaMemcpy(h,d,N*sizeof(Coord),cudaMemcpyDeviceToHost);
    uint64_t ok=0,fail=0;
    for(uint32_t i=0;i<N;i++) { if(h[i].audit==0)ok++; else fail++; }
    printf("  %-20s tpb=%-4u blk=%-6u : %7.0f M/s  iso_ok=%llu fail=%llu\n",
           label,tpb,blk,mops,(unsigned long long)ok,(unsigned long long)fail);
    cudaFree(d); free(h);
    cudaEventDestroy(t0); cudaEventDestroy(t1);
}

int main(){
    printf("=== POGLS38 World 3 GPU Bench (T4) ===\n\n");
    uint32_t Ns[]={128*1024, 512*1024, 1024*1024};
    const char *Nl[]={"128K","512K","1M"};
    for(int bi=0;bi<3;bi++){
        printf("  [batch=%s]\n",Nl[bi]);
        bench(Ns[bi], 288,  "n=4  (9 warps)");
        bench(Ns[bi], 576,  "n=8  (18 warps)");
        bench(Ns[bi], 1152, "n=16 (36 warps)");
        printf("\n");
    }
    printf("iso_fail must=0 (34n+1 mod17=1 always)\n");
    return 0;
}

In [ ]:
!nvcc -O3 -arch=sm_75 bench.cu -o bench && echo 'OK'

In [ ]:
!./bench